In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/drive/My Drive/2023 BU/2023_BU_data.txt" "/content/sample_data/2023_BU_data.txt"

Функция load_observations читает файл с астрономическими наблюдениями и парсит каждую строку:

date_ut — дата и время наблюдения

ra — прямое восхождение. Формат: часы минуты секунды

dec — склонение . Формат: градусы минуты секунды

magn — видимая яркость

location — какая обсерватория сделала наблюдение

ref — номер записи в каталоге

In [ ]:
# функция для загрузки наблюдений из текстового файла
def load_observations(filename):
    # здесь будем накапливать все считанные наблюдения в виде словарей
    observations = []

    # открываем файл на чтение в текстовом режиме с кодировкой UTF-8
    with open(filename, 'r', encoding='utf-8') as f:
        # читаем все строки файла в список
        lines = f.readlines()

        # первая строка, как правило, заголовок – пропускаем её
        data_lines = lines[1:]

        # идём по всем строчкам с данными
        for line in data_lines:
            # убираем пробелы и символ перевода строки по краям
            line = line.strip()

            # если строка пустая (например, пустая строка в конце файла) – пропускаем
            if not line:
                continue

            # разбиваем строку по пробелам
            # дополнительно удаляем пустые элементы (если между словами было несколько пробелов)
            parts = [x for x in line.split() if x]

            # если в строке меньше 12 полей – считаем, что формат неправильный, пропускаем
            if len(parts) < 12:
                continue

            # первые 3 поля – это дата в формате: ГОД МЕСЯЦ ДЕНЬ.С_ДРОБЬЮ
            # пример: "2023 01 21.34299"
            date_ut = ' '.join(parts[0:3])

            # следующие 3 поля – прямое восхождение (час, минута, секунда)
            # пример: "05 12 34.56"
            ra = ' '.join(parts[3:6])

            # далее 3 поля – склонение (градус, минута, секунда)
            # пример: "-10 23 45.6"
            dec = ' '.join(parts[6:9])

            # следующее поле – видимая звёздная величина (яркость)
            magn = parts[9]

            # location_start – индекс в массиве parts, с которого начинается описание обсерватории
            location_start = 10

            # иногда за звездной величиной идёт дополнительная буква (например, флаг качества),
            # проверяем это: если 11‑й элемент – одиночная буква (A–Z), считаем её частью magnitude
            if len(parts) > 10 and parts[10].isalpha() and len(parts[10]) == 1:
                # дописываем эту букву через пробел к строке magn
                magn += ' ' + parts[10]
                # и сдвигаем начало описания обсерватории на один элемент вправо
                location_start += 1

            # поля с location_start до предпоследних двух – это текст с названием / кодом обсерватории
            # parts[location_start:-2] – вырезка всех элементов, кроме последних двух (которые ref)
            # если там есть хотя бы один элемент – склеиваем их через пробел
            location = ' '.join(parts[location_start:-2]) if len(parts[location_start:-2]) > 0 else ''

            # последние два поля – это ref (обычно код каталога или ссылки)
            ref = ' '.join(parts[-2:])

            # если какое‑то из важнейших полей пустое – пропускаем эту строку
            if not all([date_ut, ra, dec, magn, location, ref]):
                continue

            # создаём словарь с одним наблюдением
            obs = {
                'date_ut': date_ut,   # строка с датой
                'ra': ra,             # строка с прямым восхождением
                'dec': dec,           # строка со склонением
                'magn': magn,         # звёздная величина
                'location': location, # строка с описанием обсерватории (первое слово – её код)
                'ref': ref            # строка с ссылками/примечаниями
            }

            # добавляем это наблюдение в список
            observations.append(obs)

    # возвращаем список всех корректно считанных наблюдений
    return observations


# имя файла с данными наблюдений
filename = 'sample_data/2023_BU_data.txt'

# загружаем все наблюдения из файла
obs = load_observations(filename)

# печатаем количество загруженных наблюдений
print('Число загруженных наблюдений:', len(obs))

# выводим пример первого наблюдения для проверки структуры
print('Пример наблюдения:', obs[0])


Число загруженных наблюдений: 1753
Пример наблюдения: {'date_ut': '2023 01 21.342991', 'ra': '09 50 04.192', 'dec': '+43 47 13.29', 'magn': '21.19 R', 'location': 'I41 – Palomar Mountain--ZTF', 'ref': 'MPS 1737326'}


Физические константы

In [ ]:
# подключаем numpy для работы с векторами и массивами
import numpy as np

# astropy.time.Time – удобный класс для работы со временем (JD, UTC и т.д.)
from astropy.time import Time

# SkyCoord – класс для небесных координат; get_body_barycentric_posvel – положение тел (Земли, Солнца) в барицентрической системе
from astropy.coordinates import SkyCoord, get_body_barycentric_posvel, ICRS

# модуль units – работа с физическими единицами (метры, секунды, а.е. и т.д.)
from astropy import units as u

# физическая константа скорости света
from astropy.constants import au, c

# стандартный datetime из Python – для обычного представления даты/времени
from datetime import datetime

# EarthLocation – географическое положение на Земле;
# GCRS – геоцентрическая инерциальная система координат;
# ITRS – земная (вращающаяся) система координат
from astropy.coordinates import EarthLocation, GCRS, ITRS

# get_polar_motion – функция для учёта полярного движения (здесь импортирована, но напрямую не используется)
from astropy.coordinates.builtin_frames.utils import get_polar_motion

# --- физические константы ---

# Гауссова гравитационная константа k (в радианах в день)
# по сути характеризует среднюю угловую скорость движения Земли вокруг Солнца
k = 0.01720209895  # рад/день

# гравитационный параметр Солнца µ = k^2, в единицах (а.е.^3 / день^2)
# используется в уравнениях движения для орбит в астрономических единицах
mu = k**2

# скорость света, переведённая в астрономические единицы в день (AU/day)
c_au_per_day = c.to(u.au/u.day).value

# --- инициализация времени ---

# эпоха J2000: 2000-01-01 12:00:00 UTC
# это стандартный ноль отсчёта в астрономии (JD 2451545.0)
J2000_epoch = Time('2000-01-01T12:00:00', scale='utc')

# выводим значения для контроля
print(f"Гауссова константа k = {k}")
print(f"Гравитационный параметр Солнца mu = {mu}")
print(f"Скорость света c = {c_au_per_day:.6f} AU/day")
print(f"Эпоха J2000: {J2000_epoch.iso}")

Гауссова константа k = 0.01720209895
Гравитационный параметр Солнца mu = 0.00029591220828559115
Скорость света c = 173.144633 AU/day
Эпоха J2000: 2000-01-01 12:00:00.000


In [ ]:
# словарь с координатами обсерваторий:
# ключ – код обсерватории (MPC code),
# значение – словарь с названием, долготой, широтой (в градусах) и высотой (в метрах)
observatory_coords = {
    'I41': {'name': 'Palomar Mountain--ZTF', 'lon': -116.8656, 'lat': 33.3561, 'alt': 1706},
    'L51': {'name': 'MARGO, Nauchnyi', 'lon': 34.0150, 'lat': 44.7269, 'alt': 600},
    'I93': {'name': 'St Pardon de Conques', 'lon': 2.5447, 'lat': 44.0583, 'alt': 405},
    'J95': {'name': 'Great Shefford', 'lon': -1.3225, 'lat': 51.4306, 'alt': 135},
    '858': {'name': 'Tebbutt Observatory, Edgewood', 'lon': 151.0469, 'lat': -33.8667, 'alt': 400},
    'V28': {'name': 'Deep Sky West Observatory, Rowe', 'lon': -107.3742, 'lat': 32.8311, 'alt': 1504},
    'V00': {'name': 'Kitt Peak-Bok', 'lon': -111.5967, 'lat': 31.9629, 'alt': 2071},
    'C95': {'name': 'SATINO Remote Observatory, Haute Provence', 'lon': 5.7133, 'lat': 43.9328, 'alt': 700},
    '703': {'name': 'Catalina Sky Survey', 'lon': -110.7331, 'lat': 32.4167, 'alt': 2518},
    'U52': {'name': 'Shasta Valley Observatory, Grenada', 'lon': -122.6092, 'lat': 41.6911, 'alt': 1311},
    'T05': {'name': 'ATLAS-HKO, Haleakala', 'lon': -156.2575, 'lat': 20.7069, 'alt': 3065},
    'F52': {'name': 'Pan-STARRS 2, Haleakala', 'lon': -156.2602, 'lat': 20.7075, 'alt': 3064},
    'T12': {'name': 'University of Hawaii 88-inch telescope, Maunak', 'lon': -155.4725, 'lat': 20.7069, 'alt': 4207},
    'L06': {'name': 'Sormano 2 Observatory, Bellagio Via Lattea', 'lon': 9.3711, 'lat': 45.9639, 'alt': 1331},
    'Z80': {'name': 'Northolt Branch Observatory', 'lon': -0.3850, 'lat': 51.5136, 'alt': 83},
    'Z31': {'name': 'Tenerife Observatory-LCO A, Tenerife', 'lon': -16.3222, 'lat': 28.2997, 'alt': 2364},
    '950': {'name': 'La Palma', 'lon': -17.8769, 'lat': 28.7639, 'alt': 2396},
    'G40': {'name': 'Slooh.com Canary Islands Observatory', 'lon': -17.8675, 'lat': 28.2975, 'alt': 2600},
    'P93': {'name': 'Space Tracking and Communications Center, JAXA', 'lon': 141.1186, 'lat': 35.2457, 'alt': 100},
    'C20': {'name': 'Kislovodsk Mtn. Astronomical Stn., Pulkovo Obs.', 'lon': 42.5006, 'lat': 43.7192, 'alt': 2100},
    '587': {'name': 'Sormano', 'lon': 9.3817, 'lat': 45.9639, 'alt': 1291},
    'K63': {'name': 'G. Pascoli Observatory, Barga', 'lon': 10.4889, 'lat': 44.0869, 'alt': 600},
    '203': {'name': 'GiaGa Observatory', 'lon': 11.1433, 'lat': 43.5486, 'alt': 300},
    'M11': {'name': 'Novaastro Observatory, Banon', 'lon': 5.6761, 'lat': 43.8222, 'alt': 710},
    'M33': {'name': 'OWL-Net, Mitzpe Ramon', 'lon': 34.7625, 'lat': 31.2447, 'alt': 860},
    'Z01': {'name': 'OWL-Net, Oukaimeden', 'lon': -7.6092, 'lat': 31.2069, 'alt': 2800},
    '595': {'name': "Farra d'Isonzo", 'lon': 13.5833, 'lat': 45.8042, 'alt': 50},
}

# печатаем, сколько всего обсерваторий добавлено в словарь
print(f"Загружено координат для {len(observatory_coords)} обсерваторий")

Загружено координат для 27 обсерваторий


Парсинг даты

In [ ]:
# функция для разбора даты вида "2023 01 21.34299"
# и перевода её в:
# 1) объект Time (астрономическое время)
# 2) количество секунд, прошедших от эпохи J2000
def parse_date(date_str):
    # разбиваем строку на части: год, месяц, день.с_дробью
    parts = date_str.split()

    # год
    year = int(parts[0])
    # месяц
    month = int(parts[1])
    # день с дробной частью (например, 21.34299)
    day_frac = float(parts[2])

    # целая часть – номер дня в месяце
    day = int(day_frac)
    # дробная часть – доля суток (от 0 до 1)
    fractional_day = day_frac - day

    # создаём объект datetime с датой (без времени, пока что 00:00:00)
    dt = datetime(year, month, day)

    # перевод дробной части суток в секунды:
    # 1 сутки = 86400 секунд
    seconds_in_day = fractional_day * 86400

    # часы – целая часть от деления на 3600
    hours = int(seconds_in_day // 3600)
    # минуты – целая часть от остатка, делённого на 60
    minutes = int((seconds_in_day % 3600) // 60)
    # секунды – остаток от деления на 60
    seconds = seconds_in_day % 60

    # заменяем в объекте datetime время (часы, минуты, секунды и микросекунды),
    # то есть устанавливаем точное время по UTC
    dt = dt.replace(
        hour=hours,
        minute=minutes,
        second=int(seconds),
        microsecond=int((seconds - int(seconds)) * 1e6)
    )

    # создаём объект Time из astropy с временной шкалой UTC
    t_utc = Time(dt, scale='utc')

    # считаем разницу по юлианской дате (JD) между текущим моментом и эпохой J2000
    # разность в днях умножаем на 86400, получая секунды от J2000
    dt_seconds = (t_utc.jd - J2000_epoch.jd) * 86400.0

    # возвращаем и Time‑объект, и "секунды от J2000"
    return t_utc, dt_seconds


 Парсинг координат

 Переводит координаты из текстового формата в объект SkyCoord:

RA  — даётся в часах, минутах, секундах

Dec — даётся в градусах, минутах, секундах

frame='icrs' — используем стандартную небесную систему координат ICRS

In [ ]:
# функция парсинга координат (прямое восхождение + склонение)
def parse_coords(ra_str, dec_str):
    # SkyCoord может напрямую разобрать строки:
    # ra_str – в часах (час:мин:сек), dec_str – в градусах (град:мин:сек)
    # unit=(u.hourangle, u.deg) – первая координата в часах, вторая – в градусах
    # frame='icrs' – используем систему координат ICRS (аналог J2000)
    return SkyCoord(ra_str, dec_str, unit=(u.hourangle, u.deg), frame='icrs')
    # затем из SkyCoord можно получить декартовы координаты и единичный вектор направления


In [ ]:
# функция получения позиции обсерватории в инерциальной геоцентрической системе (GCRS)
# с учётом вращения Земли
def get_observatory_position(obs_code, t_utc):
    # если код обсерватории не найден в нашем словаре
    if obs_code not in observatory_coords:
        # тогда в качестве fallback используем барицентрическое положение Земли
        # (то есть возвращаем вектор от Солнца к Земле)
        earth_pos, _ = get_body_barycentric_posvel('earth', t_utc)
        # переводим координаты в астрономические единицы и возвращаем массив
        return earth_pos.xyz.to(u.au).value

    # если код есть, вытаскиваем информацию по обсерватории
    obs_info = observatory_coords[obs_code]
    lon_deg = obs_info['lon']  # географическая долгота в градусах (восток +, запад -)
    lat_deg = obs_info['lat']  # широта в градусах (север +, юг -)
    alt_m = obs_info['alt']    # высота над уровнем моря в метрах

    # экваториальный радиус Земли (большая полуось эллипсоида WGS-84) – в метрах
    a_earth = 6378137.0
    # сплюснутость (разница между экваториальным и полярным радиусами в относительных единицах)
    f = 1.0 / 298.257223563

    # квадрат эксцентриситета эллипсоида: e^2 = 2f − f^2
    e2 = 2*f - f*f
    # синус широты
    sin_lat = np.sin(np.deg2rad(lat_deg))
    # радиус кривизны в меридиане (N) – используется при переходе к декартовым координатам
    N = a_earth / np.sqrt(1 - e2 * sin_lat**2)

    # теперь считаем прямоугольные координаты в системе ITRS (земная, связанная с поверхностью)
    cos_lat = np.cos(np.deg2rad(lat_deg))
    cos_lon = np.cos(np.deg2rad(lon_deg))
    sin_lon = np.sin(np.deg2rad(lon_deg))

    # x, y, z в ITRS (в метрах)
    # x – в направлении пересечения экватора и меридиана Greenwich
    x_itrs = (N + alt_m) * cos_lat * cos_lon
    # y – вдоль экватора на восток
    y_itrs = (N + alt_m) * cos_lat * sin_lon
    # z – по оси вращения Земли (к северному полюсу)
    z_itrs = (N * (1 - e2) + alt_m) * sin_lat

    # создаём объект ITRS (неинерциальная система, вращающаяся с Землёй)
    itrs_coord = ITRS(x=x_itrs*u.m, y=y_itrs*u.m, z=z_itrs*u.m, obstime=t_utc)

    # преобразуем координаты в GCRS (инерциальная геоцентрическая система):
    # astropy сам применяет нужные матрицы вращения, прецессии, нутации и т.п.
    gcrs_coord = itrs_coord.transform_to(GCRS(obstime=t_utc))

    # берём декартовы координаты и переводим метры в астрономические единицы (AU)
    pos_au = gcrs_coord.cartesian.xyz.to(u.au).value

    # возвращаем вектор [x, y, z] в AU относительно центра Земли
    return pos_au

Выбор трёх наблюдений для метода Гаусса

In [ ]:
# выбор 3 наблюдений для метода Гаусса
obs1_data = obs[0]
obs2_data = obs[200]
obs3_data = obs[700]

print("Выбранные наблюдения:")
print("1:", obs1_data)
print("2:", obs2_data)
print("3:", obs3_data)

Выбранные наблюдения:
1: {'date_ut': '2023 01 21.342991', 'ra': '09 50 04.192', 'dec': '+43 47 13.29', 'magn': '21.19 R', 'location': 'I41 – Palomar Mountain--ZTF', 'ref': 'MPS 1737326'}
2: {'date_ut': '2023 01 25.857638', 'ra': '09 31 54.311', 'dec': '+41 33 41.19', 'magn': '17.0', 'location': 'C20 – Kislovodsk Mtn. Astronomical Stn., Pulkovo Obs.', 'ref': 'MPS 1741295'}
3: {'date_ut': '2023 01 26.722365', 'ra': '08 59 59.361', 'dec': '+34 28 57.62', 'magn': '14.2', 'location': 'C20 – Kislovodsk Mtn. Astronomical Stn., Pulkovo Obs.', 'ref': 'MPS 1741297'}


Парсинг выбранных наблюдений

In [ ]:
# --- парсинг времени для трёх выбранных наблюдений ---

# для каждого наблюдения преобразуем строку даты в:
# 1) объект Time (t*_utc)
# 2) секунды от J2000 (t*_sec)
t1_utc, t1_sec = parse_date(obs1_data['date_ut'])
t2_utc, t2_sec = parse_date(obs2_data['date_ut'])
t3_utc, t3_sec = parse_date(obs3_data['date_ut'])

# --- парсинг координат (RA, Dec) ---

# получаем объекты SkyCoord для трёх точек
coord1 = parse_coords(obs1_data['ra'], obs1_data['dec'])
coord2 = parse_coords(obs2_data['ra'], obs2_data['dec'])
coord3 = parse_coords(obs3_data['ra'], obs3_data['dec'])

# функция, извлекающая код обсерватории из строки location
# обычно первое слово – это код MPC (например, 'I41', '703' и т.д.)
def get_obs_code(location_str):
    parts = location_str.split()
    if len(parts) > 0:
        code = parts[0]
        return code
    return None

# определяем коды обсерваторий для трёх наблюдений
obs1_code = get_obs_code(obs1_data['location'])
obs2_code = get_obs_code(obs2_data['location'])
obs3_code = get_obs_code(obs3_data['location'])

# --- геоцентрические позиции обсерваторий (в GCRS) ---

# для каждого момента времени t*_utc получаем положение обсерватории
# относительно центра Земли (в AU, в системе GCRS)
R1_vec_obs = get_observatory_position(obs1_code, t1_utc)
R2_vec_obs = get_observatory_position(obs2_code, t2_utc)
R3_vec_obs = get_observatory_position(obs3_code, t3_utc)

# --- гелиоцентрические позиции Земли ---

# получаем барицентрическое положение Земли и Солнца для каждого момента
earth_pos_1, earth_vel_1 = get_body_barycentric_posvel('earth', t1_utc)
earth_pos_2, earth_vel_2 = get_body_barycentric_posvel('earth', t2_utc)
earth_pos_3, earth_vel_3 = get_body_barycentric_posvel('earth', t3_utc)

sun_pos_1, sun_vel_1 = get_body_barycentric_posvel('sun', t1_utc)
sun_pos_2, sun_vel_2 = get_body_barycentric_posvel('sun', t2_utc)
sun_pos_3, sun_vel_3 = get_body_barycentric_posvel('sun', t3_utc)

# гелиоцентрическое положение центра Земли = положение Земли относительно барицентра
# минус положение Солнца относительно барицентра
R1_earth_helio = (earth_pos_1 - sun_pos_1).xyz.to(u.au).value
R2_earth_helio = (earth_pos_2 - sun_pos_2).xyz.to(u.au).value
R3_earth_helio = (earth_pos_3 - sun_pos_3).xyz.to(u.au).value

# полное гелиоцентрическое положение обсерватории =
# гелиоцентрическое положение Земли + положение обсерватории относительно центра Земли
R1_vec = R1_earth_helio + R1_vec_obs
R2_vec = R2_earth_helio + R2_vec_obs
R3_vec = R3_earth_helio + R3_vec_obs

# выводим время наблюдений в удобном формате и секунды от J2000
print(f"Время наблюдения 1: {t1_utc.iso}, t1_sec = {t1_sec:.2f} сек от J2000")
print(f"Время наблюдения 2: {t2_utc.iso}, t2_sec = {t2_sec:.2f} сек от J2000")
print(f"Время наблюдения 3: {t3_utc.iso}, t3_sec = {t3_sec:.2f} сек от J2000")

# считаем интервалы времени между наблюдениями в днях
dt12 = (t2_sec - t1_sec) / 86400.0
dt23 = (t3_sec - t2_sec) / 86400.0
print(f"\nВременные интервалы: Δt(1→2) = {dt12:.4f} дней, Δt(2→3) = {dt23:.4f} дней")

# печатаем коды обсерваторий для контроля
print(f"\nОбсерватории: {obs1_code}, {obs2_code}, {obs3_code}")

# печатаем векторы положений обсерваторий относительно центров Земли (в AU)
print(f"\nГелиоцентрические позиции обсерваторий (AU):")
print(f"R1_obs: {R1_vec_obs}")
print(f"R2_obs: {R2_vec_obs}")
print(f"R3_obs: {R3_vec_obs}")

Время наблюдения 1: 2023-01-21 08:13:54.422, t1_sec = 727560834.42 сек от J2000
Время наблюдения 2: 2023-01-25 20:34:59.923, t2_sec = 727950899.92 сек от J2000
Время наблюдения 3: 2023-01-26 17:20:12.336, t3_sec = 728025612.34 сек от J2000

Временные интервалы: Δt(1→2) = 4.5146 дней, Δt(2→3) = 0.8647 дней

Обсерватории: I41, C20, C20

Гелиоцентрические позиции обсерваторий (AU):
R1_obs: [-2.12868830e-05  2.85683391e-05  2.33621448e-05]
R2_obs: [-1.33946295e-05  2.77858569e-05  2.93546352e-05]
R3_obs: [1.16301609e-05 2.86264700e-05 2.92989611e-05]


Норма вектора

In [ ]:
# простая функция для вычисления евклидовой нормы вектора
def vector_norm(v):
    # сумма квадратов компонент и корень из неё
    return sum(x**2 for x in v)**0.5

Единичные векторы направления

Переводит сферические координаты (RA, Dec) в декартовы (x, y, z):

SkyCoord хранит направление на астероид

.cartesian.xyz.value — получает компоненты единичного вектора в декартовой системе

In [ ]:
# --- единичные векторы направления на астероид ---

# coord*.cartesian.xyz.value даёт декартовы координаты единичного вектора
# направления (x, y, z) в системе ICRS (по сути – направление на источник)
rho_hat1 = coord1.cartesian.xyz.value
rho_hat2 = coord2.cartesian.xyz.value
rho_hat3 = coord3.cartesian.xyz.value

# проверяем, что длина (норма) этих векторов близка к 1
print(f"Модули единичных векторов: {vector_norm(rho_hat1):.6f}, {vector_norm(rho_hat2):.6f}, {vector_norm(rho_hat3):.6f}")


Модули единичных векторов: 1.000000, 1.000000, 1.000000


Метод Гаусса для 3-х наблюдений

In [ ]:
# реализация классического метода Гаусса по трём наблюдениям
def gauss_method(
    rho_hat1, rho_hat2, rho_hat3,  # единичные векторы направлений на объект в моменты t1, t2, t3
    R1, R2, R3,                    # гелиоцентрические векторы положения обсерваторий (или Земли) в эти моменты
    t1_sec, t2_sec, t3_sec,        # времена наблюдений (в секундах от J2000)
    mu,                            # гравитационный параметр Солнца (AU^3/day^2)
    max_iter=50,                   # максимальное число итераций
    tol=1e-12                      # точность по модулю вектора r2
):
    # --- вычисляем временные интервалы в секундах ---
    tau1 = t1_sec - t2_sec  # от t2 до t1
    tau3 = t3_sec - t2_sec  # от t2 до t3
    tau  = t3_sec - t1_sec  # от t1 до t3 (полный интервал)

    # переводим интервалы времени в дни, так как формулы Лагранжа используют дни
    tau1_days = tau1 / 86400.0
    tau3_days = tau3 / 86400.0
    tau_days  = tau  / 86400.0

    print(f"\nВременные интервалы:")
    print(f"tau1 = t1 - t2 = {tau1:.2f} сек = {tau1_days:.6f} дней")
    print(f"tau3 = t3 - t2 = {tau3:.2f} сек = {tau3_days:.6f} дней")
    print(f"tau  = t3 - t1 = {tau:.2f} сек = {tau_days:.6f} дней")

    # --- вспомогательные коэффициенты D (по Гауссу) ---

    # D0 – скалярное тройное произведение трёх единичных векторов направлений
    D0  = np.dot(rho_hat1, np.cross(rho_hat2, rho_hat3))

    # D1*, D2*, D3* – скалярные тройные произведения вида (rho_i × rho_j)·Rk
    D11 = np.dot(np.cross(rho_hat2, rho_hat3), R1)
    D21 = np.dot(np.cross(rho_hat2, rho_hat3), R2)
    D31 = np.dot(np.cross(rho_hat2, rho_hat3), R3)

    D12 = np.dot(np.cross(rho_hat1, rho_hat3), R1)
    D22 = np.dot(np.cross(rho_hat1, rho_hat3), R2)
    D32 = np.dot(np.cross(rho_hat1, rho_hat3), R3)

    D13 = np.dot(np.cross(rho_hat1, rho_hat2), R1)
    D23 = np.dot(np.cross(rho_hat1, rho_hat2), R2)
    D33 = np.dot(np.cross(rho_hat1, rho_hat2), R3)

    print(f"\nD-коэффициенты:")
    print(f"D0 = {D0:.6f}")
    print(f"D11, D12, D13 = {D11:.6f}, {D12:.6f}, {D13:.6f}")
    print(f"D21, D22, D23 = {D21:.6f}, {D22:.6f}, {D23:.6f}")
    print(f"D31, D32, D33 = {D31:.6f}, {D32:.6f}, {D33:.6f}")

    # --- начальное приближение для расстояния до объекта во втором наблюдении r2_mag ---

    # берём просто модуль гелиоцентрического вектора обсерватории во время t2 (как грубую оценку)
    r2_mag = vector_norm(R2)  # в AU
    r2_mag_old = 0.0          # сюда будем сохранять предыдущее значение для проверки сходимости

    print(f"\nНачальное приближение r2 = {r2_mag:.6f} AU")

    # --- итерационный процесс ---
    for iteration in range(max_iter):
        print(f"\nИтерация {iteration + 1}")

        # проверяем сходимость: если r2 почти не меняется – выходим
        if abs(r2_mag - r2_mag_old) < tol:
            print(f"Сходимость достигнута на итерации {iteration + 1}")
            break

        # запоминаем старое значение перед перерасчётом
        r2_mag_old = r2_mag

        # коэффициенты Лагранжа f и g (первое приближение, разложение в ряд по времени)
        f1 = 1.0 - 0.5 * mu * tau1_days**2 / r2_mag**3
        f3 = 1.0 - 0.5 * mu * tau3_days**2 / r2_mag**3
        g1 = tau1_days - (1.0/6.0) * mu * tau1_days**3 / r2_mag**3
        g3 = tau3_days - (1.0/6.0) * mu * tau3_days**3 / r2_mag**3

        # знаменатель в формулах для коэффициентов c1 и c3
        denominator = f1 * g3 - f3 * g1
        if abs(denominator) < 1e-15:
            print("Ошибка: знаменатель слишком мал")
            break

        # коэффициенты c1 и c3 из уравнений Гаусса
        c1 = g3 / denominator
        c3 = -g1 / denominator

        # --- геоцентрические расстояния rho1, rho2, rho3 (модуль векторов от обсерватории до объекта) ---

        rho1 = (-D11 * c1 + D21 - D31 * c3) / D0
        rho2 = (-D12 * c1 + D22 - D32 * c3) / D0
        rho3 = (-D13 * c1 + D23 - D33 * c3) / D0

        # --- гелиоцентрические векторы положения объекта в моменты t1, t2, t3 ---

        # r_i = R_i (положение обсерватории) + rho_i * rho_hat_i (направление на объект)
        r1_vec = R1 + rho1 * rho_hat1
        r2_vec = R2 + rho2 * rho_hat2
        r3_vec = R3 + rho3 * rho_hat3

        # новый модуль r2 (для проверки сходимости)
        r2_mag_new = vector_norm(r2_vec)

        # --- учёт времени распространения света (light-time correction) ---

        # время, за которое свет проходит расстояние rho_i (в секундах)
        light_time1 = rho1 / c_au_per_day * 86400.0
        light_time2 = rho2 / c_au_per_day * 86400.0
        light_time3 = rho3 / c_au_per_day * 86400.0

        # корректируем времена наблюдений, вычитая время распространения света
        t1_corrected = t1_sec - light_time1
        t2_corrected = t2_sec - light_time2
        t3_corrected = t3_sec - light_time3

        # пересчитываем интервалы времени с учётом светового запаздывания
        tau1 = t1_corrected - t2_corrected
        tau3 = t3_corrected - t2_corrected
        tau  = t3_corrected - t1_corrected

        # снова переводим интервалы в дни
        tau1_days = tau1 / 86400.0
        tau3_days = tau3 / 86400.0
        tau_days  = tau  / 86400.0

        # обновляем r2_mag для следующей итерации
        r2_mag = r2_mag_new

        print(f"rho = [{rho1:.6f}, {rho2:.6f}, {rho3:.6f}] AU")
        print(f"r2_mag = {r2_mag:.6f} AU, Δ = {abs(r2_mag - r2_mag_old):.2e}")
        print(f"light_time = [{light_time1:.2f}, {light_time2:.2f}, {light_time3:.2f}] сек")

    else:
        # этот блок выполняется, если цикл завершился не через break, а по достижению max_iter
        print(f"Достигнуто максимальное число итераций ({max_iter})")

    # возвращаем векторы r1, r2, r3, расстояния rho1..rho3 и интервалы времени tau1_days, tau3_days
    return r1_vec, r2_vec, r3_vec, rho1, rho2, rho3, tau1_days, tau3_days


# вызываем метод Гаусса с подготовленными данными
r1_vec, r2_vec, r3_vec, rho1_mag, rho2_mag, rho3_mag, tau1, tau3 = gauss_method(
    rho_hat1, rho_hat2, rho_hat3,  # единичные направления
    R1_vec, R2_vec, R3_vec,        # гелиоцентрические позиции обсерваторий
    t1_sec, t2_sec, t3_sec,        # времена наблюдений (сек от J2000)
    mu                             # гравитационный параметр Солнца
)

# печатаем получившиеся гелиоцентрические векторы положения объекта в трёх моментах
print("\nИтоговые гелиоцентрические векторы астероида (AU):")
print("r1:", r1_vec)
print("r2:", r2_vec)
print("r3:", r3_vec)
print(f"Модули r: {vector_norm(r1_vec):.6f}, {vector_norm(r2_vec):.6f}, {vector_norm(r3_vec):.6f}")



Временные интервалы:
tau1 = t1 - t2 = -390065.50 сек = -4.514647 дней
tau3 = t3 - t2 = 74712.41 сек = 0.864727 дней
tau  = t3 - t1 = 464777.91 сек = 5.379374 дней

D-коэффициенты:
D0 = 0.002126
D11, D12, D13 = -0.000320, 0.003485, 0.005738
D21, D22, D23 = 0.011765, 0.020166, 0.010452
D31, D32, D33 = 0.014070, 0.023345, 0.011346

Начальное приближение r2 = 0.984567 AU

Итерация 1
rho = [0.001358, 0.001889, 0.000557] AU
r2_mag = 0.986264 AU, Δ = 1.70e-03
light_time = [0.68, 0.94, 0.28] сек

Итерация 2
rho = [0.001379, 0.001924, 0.000574] AU
r2_mag = 0.986296 AU, Δ = 3.15e-05
light_time = [0.69, 0.96, 0.29] сек

Итерация 3
rho = [0.001379, 0.001925, 0.000575] AU
r2_mag = 0.986296 AU, Δ = 5.40e-07
light_time = [0.69, 0.96, 0.29] сек

Итерация 4
rho = [0.001379, 0.001925, 0.000575] AU
r2_mag = 0.986296 AU, Δ = 9.19e-09
light_time = [0.69, 0.96, 0.29] сек

Итерация 5
rho = [0.001379, 0.001925, 0.000575] AU
r2_mag = 0.986296 AU, Δ = 1.57e-10
light_time = [0.69, 0.96, 0.29] сек

Итерация 6
rh

Вычисление вектора скорости

In [ ]:
# --- вычисление вектора скорости v2 по трём положениям r1, r2, r3 ---

# модуль вектора r2 (гелиоцентрическое расстояние объекта во втором наблюдении)
r2_mag = vector_norm(r2_vec)

# пересчёт коэффициентов Лагранжа f и g для итоговых tau1, tau3 и r2_mag
f1 = 1.0 - 0.5 * mu * tau1**2 / r2_mag**3
f3 = 1.0 - 0.5 * mu * tau3**2 / r2_mag**3
g1 = tau1 - (1.0/6.0) * mu * tau1**3 / r2_mag**3
g3 = tau3 - (1.0/6.0) * mu * tau3**3 / r2_mag**3

# знаменатель в формулах для скорости
denominator = f1 * g3 - f3 * g1

# формула Гаусса для нахождения скорости в момент t2:
# v2 = (-f3*r1 + f1*r3) / (f1*g3 - f3*g1)
v2_vec = (-f3 * r1_vec + f1 * r3_vec) / denominator

print("Гелиоцентрический вектор положения r2 (AU):", r2_vec)
print("Гелиоцентрический вектор скорости v2 (AU/день):", v2_vec)
print(f"Модуль скорости |v2| = {vector_norm(v2_vec):.6f} AU/день")


Гелиоцентрический вектор положения r2 (AU): [-0.56990337  0.73823264  0.32094108]
Гелиоцентрический вектор скорости v2 (AU/день): [-0.01422765 -0.00921911 -0.00409659]
Модуль скорости |v2| = 0.017441 AU/день


Расчёт орбитальных элементов (Метод Гаусса)

Угловой момент h = r × v

Характеризует вращение орбиты

h_z — компонента, определяющая наклонение орбиты

Наклонение i = arccos(h_z / h)

Угол между плоскостью орбиты и эклиптикой (плоскость орбиты Земли)

Эксцентриситет e

Характеризует вытянутость орбиты

Большая полуось a

Половина максимального размера орбиты
Определяется энергией орбиты

In [ ]:
# --- расчёт орбитальных элементов по найденным r2 и v2 ---

# модуль радиуса-вектора r
r_mag = vector_norm(r2_vec)
# модуль скорости
v_mag = vector_norm(v2_vec)

# вектор углового момента h = r × v
h_vec = np.cross(r2_vec, v2_vec)
# его модуль
h_mag = vector_norm(h_vec)

# наклонение орбиты i – угол между вектором h и осью z:
# cos(i) = h_z / |h|
i = np.arccos(np.clip(h_vec[2] / h_mag, -1, 1))

# вектор линии восходящего узла: n = k × h, где k = (0, 0, 1)
# (хотя далее n_vec может не использоваться, классически он нужен для вычисления Ω)
n_vec = np.cross([0, 0, 1], h_vec)
n_mag = vector_norm(n_vec)

# вектор эксцентриситета:
# e_vec = (v × h)/µ − r/|r|
e_vec = (np.cross(v2_vec, h_vec) / mu) - (r2_vec / r_mag)
# модуль эксцентриситета
e = vector_norm(e_vec)

# удельная энергия орбиты: ε = v^2/2 − µ/r
energy = v_mag**2 / 2.0 - mu / r_mag

# из энергии получаем большую полуось:
# a = −µ / (2ε)  (для эллиптической орбиты ε < 0)
if abs(energy) < 1e-15:
    print("ОШИБКА: большая полуось близка к нулю")
    a = float('inf')
else:
    a = -mu / (2.0 * energy)

print("\nРАССЧИТАННЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ")
print(f"Большая полуось (a):                {a:.6f} AU")
print(f"Эксцентриситет (e):                 {e:.6f}")
print(f"Наклонение (i):                     {np.rad2deg(i):.4f}°")

# --- «реальные» элементы орбиты для сравнения (например, из JPL) ---

print("\nРЕАЛЬНЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ")
real_elements = {
    'a': 1.02108734,  # большая полуось
    'e': 0.032101,    # эксцентриситет
    'i': 22.37654     # наклонение в градусах
}
print(f"Большая полуось (a):                {real_elements['a']:.6f} AU")
print(f"Эксцентриситет (e):                 {real_elements['e']:.6f}")
print(f"Наклонение (i):                     {real_elements['i']:.4f}°")

# --- относительные ошибки в процентах ---

print("\nОШИБКИ РАСЧЕТА (%)")
if a != float('inf'):
    # ошибка по большой полуоси
    error_a = abs(a - real_elements['a']) / real_elements['a'] * 100
    print(f"Большая полуось (a): {error_a:.2f}%")
else:
    print(f"Большая полуось (a): не определена")

# относительная ошибка по эксцентриситету
error_e = abs(e - real_elements['e']) / real_elements['e'] * 100
# относительная ошибка по наклонению
error_i = abs(np.rad2deg(i) - real_elements['i']) / real_elements['i'] * 100
print(f"Эксцентриситет (e): {error_e:.2f}%")
print(f"Наклонение (i): {error_i:.2f}%")



РАССЧИТАННЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ
Большая полуось (a):                1.000220 AU
Эксцентриситет (e):                 0.013939
Наклонение (i):                     23.6519°

РЕАЛЬНЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ
Большая полуось (a):                1.021087 AU
Эксцентриситет (e):                 0.032101
Наклонение (i):                     22.3765°

ОШИБКИ РАСЧЕТА (%)
Большая полуось (a): 2.04%
Эксцентриситет (e): 56.58%
Наклонение (i): 5.70%


Подготовка данных для Гаусса-Ньютона

In [ ]:
# --- подготовка набора наблюдений вокруг t2 для уточнения орбиты методом Гаусса–Ньютона ---

# окно по времени: берём наблюдения в пределах ±1 суток от момента t2
window_seconds = 86400.0  # 1 день в секундах

# опорное время – второе наблюдение (t2_sec)
t2_sec_ref = t2_sec
t_min = t2_sec_ref - window_seconds  # нижняя граница
t_max = t2_sec_ref + window_seconds  # верхняя граница

# сюда будем складывать времена, координаты и положения обсерваторий
times_all_sec = []        # времена наблюдений (сек от J2000)
coords_all = []           # объекты SkyCoord (не обязательно используются далее)
rho_hat_all_list = []     # единичные векторы направления на объект
earth_sun_all_list = []   # гелиоцентрические позиции обсерваторий
observatory_all_list = [] # позиции обсерваторий в GCRS (относительно центра Земли)

# проходим по всем наблюдениям из файла
for o in obs:
    # парсим дату, получаем время и секунды от J2000
    t_utc, t_sec = parse_date(o['date_ut'])

    # отбираем только те наблюдения, которые попадают в окно [t_min, t_max]
    if t_sec < t_min or t_sec > t_max:
        continue

    # парсим координаты RA/Dec
    coord = parse_coords(o['ra'], o['dec'])
    # единичный вектор направления в декартовых координатах
    rho_hat = coord.cartesian.xyz.value

    # получаем барицентрическое положение Земли и Солнца
    earth_pos, earth_vel = get_body_barycentric_posvel('earth', t_utc)
    sun_pos, sun_vel = get_body_barycentric_posvel('sun', t_utc)

    # гелиоцентрическая позиция Земли
    R_earth = (earth_pos - sun_pos).xyz.to(u.au).value

    # код обсерватории (первое слово из поля location)
    obs_code = get_obs_code(o['location'])

    # позиция обсерватории в GCRS (от центра Земли)
    R_obs = get_observatory_position(obs_code, t_utc)

    # гелиоцентрическая позиция обсерватории =
    # гелиоцентрическая позиция Земли + вектор от Земли до обсерватории
    R_total = R_earth + R_obs

    # сохраняем всё в соответствующие списки
    times_all_sec.append(t_sec)
    coords_all.append(coord)
    rho_hat_all_list.append(rho_hat)
    earth_sun_all_list.append(R_total)
    observatory_all_list.append(R_obs)

# переводим списки в numpy-массивы
times_all_sec = np.array(times_all_sec)
rho_hat_all = np.array(rho_hat_all_list)
earth_sun_all = np.array(earth_sun_all_list)      # гелиоцентрические позиции обсерваторий
observatory_all = np.array(observatory_all_list)  # позиции обсерваторий относительно центра Земли

# опорное время для интегрирования – t2 (в секундах от J2000)
t_ref = t2_sec

# dt_all_sec – сдвиг всех наблюдений относительно t_ref
dt_all_sec = times_all_sec - t_ref
# переводим в дни (нужно для орбитального интегратора)
dt_all_days = dt_all_sec / 86400.0

# начальное приближение для вектора параметров (орбиты) x0:
# первые 3 компоненты – r2_vec (положение),
# следующие 3 – v2_vec (скорость):
x0 = np.hstack((r2_vec, v2_vec))


Кеплеровский интегратор

Определяем параметры орбиты (большая полуось, эксцентриситет)

Вычисляем среднюю аномалию M = n·dt (n = угловая частота)

Решаем уравнение Кеплера: E - e·sin(E) = M

По истинной аномалии ν вычисляем новое положение

In [ ]:
# функция Кеплеровского интегрирования:
# по начальному положению r0 и скорости v0 в момент t_ref
# вычисляет положение r и скорость v через время dt_days (в днях)
def kepler_propagate(r0, v0, dt_days, mu):
    # приводим входные векторы к numpy-массивам типа float
    r0 = np.array(r0, dtype=float)
    v0 = np.array(v0, dtype=float)

    # модуль начального радиуса и скорости
    r0_norm = vector_norm(r0)
    v0_norm = vector_norm(v0)

    # если радиус почти нулевой (некорректная ситуация) – просто возвращаем исходные векторы
    if r0_norm < 1e-12:
        return r0, v0

    # удельная орбитальная энергия ε = v^2/2 − µ/r
    energy = v0_norm ** 2 / 2.0 - mu / r0_norm

    # если энергия >= 0 – орбита неэллиптическая (параболическая или гиперболическая),
    # а эта реализация рассчитана на эллиптический случай – просто возвращаем исходные
    if energy >= 0:
        return r0, v0

    # большая полуось эллиптической орбиты: a = −µ / (2ε)
    a = -mu / (2.0 * energy)

    # вектор углового момента h = r0 × v0
    h_vec = np.cross(r0, v0)
    # модуль углового момента
    h = vector_norm(h_vec)

    # если угловой момент почти нулевой – орбита вырождается (радиальная),
    # интегратор в таком виде не работает
    if h < 1e-12:
        return r0, v0

    # вектор эксцентриситета e_vec = (v × h)/µ − r/|r|
    e_vec = (np.cross(v0, h_vec) / mu) - r0 / r0_norm
    # модуль эксцентриситета
    e = vector_norm(e_vec)

    # ограничиваем эксцентриситет сверху (например, чтобы не попасть в крайности e ≈ 1)
    if e >= 0.99:
        e = 0.99
        # нормируем вектор эксцентриситета и масштабируем до 0.99
        e_vec = e_vec * (e / vector_norm(e_vec))

    # --- особый случай: почти круговая орбита (e ~ 0) ---
    if e < 1e-8:
        # среднее движение: n = sqrt(µ / a^3)
        n = np.sqrt(mu / a ** 3)
        # средняя аномалия за интервал dt_days: M = n * dt
        M = n * dt_days

        # базис в плоскости орбиты:
        # k_vec – ось z
        k_vec = np.array([0.0, 0.0, 1.0])
        # i_vec – направление пересечения плоскости орбиты с плоскостью экватора
        i_vec = np.cross(k_vec, h_vec)
        i_norm = np.linalg.norm(i_vec)
        if i_norm < 1e-15:
            # если орбита почти экваториальная, выбираем произвольное направление
            i_vec = np.array([1.0, 0.0, 0.0])
        else:
            i_vec /= i_norm
        # j_vec – орт, дополняющий базис в плоскости орбиты
        j_vec = np.cross(h_vec/h, i_vec)

        # проецируем r0 на базис (i_vec, j_vec)
        x0 = np.dot(r0, i_vec)
        y0 = np.dot(r0, j_vec)
        # радиус в плоскости орбиты
        r = np.sqrt(x0**2 + y0**2)
        # начальный угол phi0 = atan2(y0, x0)
        phi0 = np.arctan2(y0, x0)
        # новый угол после смещения по средней аномалии
        phi = phi0 + M

        # новое положение – тот же радиус r, только угол phi
        r_vec = r * (np.cos(phi)*i_vec + np.sin(phi)*j_vec)
        # скорость оставляем как была (упрощённое приближение)
        return r_vec, v0

    # --- общий эллиптический случай 0 < e < 1 ---

    # истинная аномалия nu0 в начальный момент:
    # cos(nu0) = (e_vec · r0) / (e * |r0|)
    cos_nu0 = np.dot(e_vec, r0) / (e * r0_norm)
    cos_nu0 = np.clip(cos_nu0, -1.0, 1.0)
    nu0 = np.arccos(cos_nu0)
    # если скалярное произведение r0·v0 < 0, объект движется "назад" по орбите,
    # корректируем nu0
    if np.dot(r0, v0) < 0:
        nu0 = 2 * np.pi - nu0

    # эксцентриситетная аномалия E0 через истинную аномалию nu0:
    # tan(E0/2) = tan(nu0/2) * sqrt((1−e)/(1+e))
    arg = (1 - e) / (1 + e)
    if arg <= 0:
        # защита от численных проблем
        return r0, v0

    E0 = 2 * np.arctan(np.tan(nu0 / 2) * np.sqrt(arg))
    # начальная средняя аномалия M0 = E0 − e*sin(E0)
    M0 = E0 - e * np.sin(E0)

    # проверка, что большая полуось положительна (для эллипса)
    if a <= 0:
        return r0, v0

    # среднее движение n = sqrt(µ / a^3)
    n = np.sqrt(mu / a ** 3)
    # средняя аномалия через dt_days: M = M0 + n * dt
    M = M0 + n * dt_days

    # --- решаем уравнение Кеплера: E − e*sin(E) = M ---

    # берём E ≈ M как начальное приближение
    E = M.copy()
    for _ in range(30):
        # значение уравнения и его производная
        f = E - e * np.sin(E) - M
        fp = 1 - e * np.cos(E)
        # шаг Ньютона
        dE = -f / fp
        E += dE
        # если поправка мала – считаем, что сошлись
        if np.max(np.abs(dE)) < 1e-12:
            break

    # расстояние r = a (1 − e cos(E))
    cos_E = np.cos(E)
    r = a * (1 - e * cos_E)

    # --- базис в плоскости орбиты ---

    k_vec = np.array([0.0, 0.0, 1.0])
    i_vec = np.cross(k_vec, h_vec)
    i_norm = vector_norm(i_vec)
    if i_norm < 1e-15:
        i_vec = np.array([1.0, 0.0, 0.0])
    else:
        i_vec /= i_norm
    j_vec = np.cross(h_vec/h, i_vec)

    # направление главной оси орбиты (перицентр) – направление вектора эксцентриситета
    if e > 1e-10:
        p_dir = e_vec / e
    else:
        p_dir = i_vec
    # q_dir – второе направление в плоскости орбиты, перпендикулярное p_dir
    q_dir = np.cross(h_vec / h, p_dir)

    # истинная аномалия nu через эксцентриситетную аномалию E:
    # tan(nu/2) = sqrt((1+e)/(1−e)) * tan(E/2)
    factor = (1 + e) / (1 - e)
    if factor <= 0:
        return r0, v0
    nu = 2 * np.arctan2(np.sqrt(factor) * np.sin(E/2), np.cos(E/2))

    # радиус-вектор: r_vec = r * (cos(nu)*p_dir + sin(nu)*q_dir)
    r_vec = r * (np.cos(nu) * p_dir + np.sin(nu) * q_dir)

    # радиальная и поперечная компоненты скорости
    rdot = (mu / h) * e * np.sin(nu)
    rfdot = (mu / h) * (1 + e * np.cos(nu))
    # полная скорость: v = rdot*(r/|r|) + rfdot*q_dir
    v_vec = rdot * (r_vec / r) + rfdot * q_dir

    # возвращаем новые положение и скорость
    return r_vec, v_vec


Моделирование направления

По известному положению и скорости астероида в момент t2, вычисляет ожидаемое направление на астероид в момент ti

In [ ]:
# функция, которая по параметрам орбиты x (r0, v0),
# временному сдвигу dt_days и положению обсерватории R_obs_i
# возвращает ожидаемый единичный вектор направления на объект
def model_direction(x, dt_days, R_obs_i):
    # первые 3 компоненты x – начальный радиус-вектор r0
    r0 = x[:3]
    # следующие 3 – начальная скорость v0
    v0 = x[3:]

    # по Кеплеровскому интегратору получаем положение объекта через dt_days
    r_obj, v_obj = kepler_propagate(r0, v0, dt_days, mu)

    # вектор от обсерватории до объекта:
    # rho_vec = r_obj (гелиоцентрическое положение объекта) − R_obs_i (гелиоцентрическое положение обсерватории)
    rho_vec = r_obj - R_obs_i

    # возвращаем нормированный вектор направления (единичный вектор)
    return rho_vec / vector_norm(rho_vec)


Невязки

In [ ]:
# функция, вычисляющая вектор невязок для метода Гаусса–Ньютона
# x – текущий вектор параметров (r0, v0) в момент t_ref
def residuals(x):
    # количество наблюдений N
    N = len(dt_all_days)

    # каждой точке соответствуют 3 компоненты невязки (по x, y, z направления),
    # поэтому итоговый вектор размера 3N
    res = np.zeros(3*N)

    # для каждого наблюдения
    for i in range(N):
        # моделируем направление на объект в момент наблюдения:
        # dt_all_days[i] – сдвиг по времени (в днях) относительно t_ref,
        # earth_sun_all[i] – гелиоцентрическое положение обсерватории
        d_model = model_direction(x, dt_all_days[i], earth_sun_all[i])

        # наблюдаемое направление (единичный вектор) из данных
        d_obs = rho_hat_all[i]

        # невязка = (модельное направление) − (наблюдаемое направление)
        r_i = d_model - d_obs

        # кладём три компоненты в общий вектор res
        res[3*i:3*i+3] = r_i

    # возвращаем вектор всех невязок
    return res


Матрица Якоби

Вычисление матрицы Якоби показывает, как изменяются невязки при небольших изменениях параметров

Численное дифференцирование:

Вычисляем невязки при x+eps и x-eps
Разница деленная на 2eps ≈ производная

In [ ]:
# численное вычисление матрицы Якоби J = ∂r/∂x
# по сути считаем производные невязок по параметрам с помощью конечных разностей
def jacobian(x, eps=1e-6):
    # сначала считаем невязки при текущем x
    r0 = residuals(x)
    m = r0.size   # число строк (3N)
    n = x.size    # число параметров (8: 3 координаты r0 + 3 координаты v0)

    # создаём матрицу J размера m×n
    J = np.zeros((m, n))

    # для каждого параметра x_j
    for j in range(n):
        # создаём вектор приращения dx, где только j‑я компонента равна eps
        dx = np.zeros_like(x)
        dx[j] = eps

        # считаем невязки при x + dx
        r_plus = residuals(x + dx)
        # и при x − dx
        r_minus = residuals(x - dx)

        # производная по x_j ≈ (r(x+dx) − r(x−dx)) / (2 eps)
        J[:, j] = (r_plus - r_minus) / (2*eps)

    # возвращаем матрицу Якоби
    return J


LUP-разложение

In [ ]:
# LUP-разложение матрицы A:
# A = P^T * L * U, где:
# L – нижнетреугольная матрица с единицами на диагонали,
# U – верхнетреугольная,
# P – матрица перестановок (частичный поворот).
def LUP_decomposition(A):
    n = len(A)
    # создаём копию A как массив с плавающей точкой
    A = np.array(A, dtype=float)

    # P – единичная матрица (поначалу без перестановок)
    P = np.eye(n)

    # L и U инициализируем нулями
    L = np.zeros((n, n))
    U = np.zeros((n, n))

    # основной цикл по столбцам k
    for k in range(n):
        # выбираем строку с максимальным по модулю элементом в столбце k ниже диагонали
        pivot_row = k + np.argmax(np.abs(A[k:, k]))

        # если опорная строка не совпадает с текущей – меняем местами
        if pivot_row != k:
            A[[k, pivot_row]] = A[[pivot_row, k]]
            P[[k, pivot_row]] = P[[pivot_row, k]]
            # если уже есть ненулевые элементы в L слева от диагонали – тоже переставляем
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]

        # если диагональный элемент очень мал – матрица почти вырождена
        if abs(A[k, k]) < 1e-12:
            raise ValueError("Матрица вырождена или почти вырождена")

        # на диагонали L всегда стоят единицы
        L[k, k] = 1.0

        # k‑я строка U – это текущая строка A справа от диагонали
        U[k, k:] = A[k, k:]

        # ниже диагонали в столбце k элементы L[i,k] = A[i,k]/A[k,k]
        L[k+1:, k] = A[k+1:, k] / A[k, k]

        # вычитаем влияние k‑й строки из остальных строк (гауссово исключение)
        for i in range(k+1, n):
            for j in range(k+1, n):
                A[i, j] -= L[i, k] * U[k, j]

    # возвращаем матрицы L, U и P
    return L, U, P


# решение системы LUP * x = b
def LUP_solve(L, U, P, b):
    n = len(L)

    # сначала перемножаем P и b (учёт перестановок строк)
    Pb = np.dot(P, b)

    # прямой ход: решаем L * y = P * b
    y = np.zeros(n)
    for i in range(n):
        # y[i] = Pb[i] − сумма L[i, :i] * y[:i]
        y[i] = Pb[i] - np.dot(L[i, :i], y[:i])

    # обратный ход: решаем U * x = y
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        # x[i] = (y[i] − сумма U[i, i+1:] * x[i+1:]) / U[i, i]
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]

    # получаем решение x
    return x


# удобная обёртка для решения A*x = b за один вызов
def solve_system(A, b):
    L, U, P = LUP_decomposition(A)
    return LUP_solve(L, U, P, b)


Метод Гаусса-Ньютона с регуляризацией

Это итерационный метод оптимизации для уточнения орбиты. Он уменьшает невязки, двигая параметры в правильном направлении.

Алгоритм:

Вычисляем невязки: r = model - observations

Вычисляем матрицу Якоби: J = ∂r/∂x

Решаем систему: (J^T·J)·dx = -J^T·r

Это система, которая даёт поправку dx

Ограничиваем шаг: чтобы не прыгнуть слишком далеко

Обновляем: x_new = x + dx

Повторяем, пока dx не станет очень маленькой

Регуляризация Тихонова:

"Штрафует" большие отклонения от начального приближения

Делает решение более устойчивым

In [ ]:
# реализация метода Гаусса–Ньютона с регуляризацией Тихонова
# для уточнения вектора параметров x (r0, v0)
def gauss_newton_tikhonov(x0, lambda_reg=0.3, max_iter=5, tol=1e-8):
    # начальное приближение x
    x = x0.copy()

    # матрица L для регуляризации Тихонова
    # здесь более сильно штрафуются отклонения по z-компонентам (коэффициент 10)
    L = np.diag([
        1.0, 1.0, 10.0,   # веса для r_x, r_y, r_z
        1.0, 1.0, 10.0    # веса для v_x, v_y, v_z
    ])

    # основной итерационный цикл
    for k_iter in range(max_iter):
        # проверка на NaN/inf в x – если есть, останавливаемся
        if not np.all(np.isfinite(x)):
            print("x содержит NaN/inf, останавливаемся")
            break

        # считаем вектор невязок r(x)
        r = residuals(x)
        # и матрицу Якоби J(x)
        J = jacobian(x)

        # формируем матрицу нормальных уравнений с регуляризацией:
        # A = J^T * J + λ^2 * L^T * L
        A = np.dot(J.T, J) + (lambda_reg**2) * np.dot(L.T, L)

        # правая часть:
        # b = -J^T * r − λ^2 * L^T * L * (x − x0)
        # вторая часть отвечает за "притягивание" решения к начальному приближению x0
        b = -np.dot(J.T, r) - (lambda_reg**2) * np.dot(np.dot(L.T, L), (x - x0))

        try:
            # решаем систему A * dx = b
            dx = solve_system(A, b)
        except ValueError as e:
            # если матрица вырождена – останавливаем процесс
            print(f"Ошибка при решении системы: {e}")
            break

        # ограничиваем максимальный шаг, чтобы не делать слишком большие скачки
        max_step = 0.01
        norm_dx = vector_norm(dx)
        if norm_dx > max_step:
            # масштабируем dx так, чтобы его норма была max_step
            dx = dx * (max_step / norm_dx)

        # новое значение параметров
        x_new = x + dx

        # проверяем, что не возникли NaN/inf
        if not np.all(np.isfinite(x_new)):
            print("dx привел к NaN/inf, уменьши шаг или lambda_reg")
            break

        # критерий сходимости: если шаг dx стал меньше tol – считаем, что сошлись
        if vector_norm(dx) < tol:
            print(f"Гаусс–Ньютон сошелся за {k_iter+1} итераций")
            return x_new

        # обновляем x для следующей итерации
        x = x_new

    print("Гаусс–Ньютон: достигнуто max_iter")
    return x




In [ ]:
# ещё раз формируем начальное приближение (на случай, если изменялось)
x0 = np.hstack((r2_vec, v2_vec))

# запускаем уточнение орбиты
x_refined = gauss_newton_tikhonov(
    x0,
    lambda_reg=0.3,  # коэффициент регуляризации
    max_iter=5,      # максимум итераций
    tol=1e-8         # порог по величине шага
)

# выделяем уточнённые векторы положения и скорости
r2_vec_refined = x_refined[:3]
v2_vec_refined = x_refined[3:]

print("Уточненный r2:", r2_vec_refined)
print("Уточненный v2:", v2_vec_refined)

Гаусс–Ньютон: достигнуто max_iter
Уточненный r2: [-0.56907623  0.73759393  0.32003281]
Уточненный v2: [-0.01451132 -0.0091121  -0.00375261]


Орбитальные элементы после уточнения

In [ ]:
# --- пересчёт орбитальных элементов по уточнённым r2 и v2 ---

# удобные обозначения
r2_ref = r2_vec_refined
v2_ref = v2_vec_refined

# модуль радиуса и скорости
r_mag = np.linalg.norm(r2_ref)
v_mag = np.linalg.norm(v2_ref)

# угловой момент h = r × v
h_vec = np.cross(r2_ref, v2_ref)
h_mag = np.linalg.norm(h_vec)

# наклонение орбиты
i = np.arccos(np.clip(h_vec[2] / h_mag, -1, 1))

# вектор эксцентриситета
e_vec = (np.cross(v2_ref, h_vec) / mu) - (r2_ref / r_mag)
e = np.linalg.norm(e_vec)

# удельная энергия
energy = v_mag**2 / 2.0 - mu / r_mag
if abs(energy) < 1e-15:
    print("ОШИБКА: большая полуось близка к нулю")
    a = float('inf')
else:
    # большая полуось
    a = -mu / (2.0 * energy)

print("\nРАССЧИТАННЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ")
print(f"Большая полуось (a): {a:.6f} AU")
print(f"Эксцентриситет (e): {e:.6f}")
print(f"Наклонение (i): {np.rad2deg(i):.4f}°")

print("\nРЕАЛЬНЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ")
real_elements = {
    'a': 1.02108734,
    'e': 0.032101,
    'i': 22.37654
}
print(f"Большая полуось (a): {real_elements['a']:.6f} AU")
print(f"Эксцентриситет (e): {real_elements['e']:.6f}")
print(f"Наклонение (i): {real_elements['i']:.4f}°")

print("\nОШИБКИ РАСЧЕТА (%)")
if a != float('inf'):
    error_a = abs(a - real_elements['a']) / real_elements['a'] * 100
    print(f"Большая полуось (a): {error_a:.2f}%")
else:
    print("Большая полуось (a): не определена")

error_e = abs(e - real_elements['e']) / real_elements['e'] * 100
error_i = abs(np.rad2deg(i) - real_elements['i']) / real_elements['i'] * 100
print(f"Эксцентриситет (e): {error_e:.2f}%")
print(f"Наклонение (i): {error_i:.2f}%")



РАССЧИТАННЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ
Большая полуось (a): 1.009530 AU
Эксцентриситет (e): 0.031086
Наклонение (i): 23.1123°

РЕАЛЬНЫЕ ОРБИТАЛЬНЫЕ ЭЛЕМЕНТЫ
Большая полуось (a): 1.021087 AU
Эксцентриситет (e): 0.032101
Наклонение (i): 22.3765°

ОШИБКИ РАСЧЕТА (%)
Большая полуось (a): 1.13%
Эксцентриситет (e): 3.16%
Наклонение (i): 3.29%


# Детальный разбор функции `kepler_propagate` с формулами и геометрическим смыслом

## Обзор

Функция `kepler_propagate` решает **задачу Кеплера**: по известному положению **r₀** и скорости **v₀** объекта в момент времени t = 0, вычислить его положение **r** и скорость **v** через интервал времени Δt.

Это основано на том, что в гравитационном поле Солнца объект движется по конической секции (в нашем случае — по эллипсу). Траектория полностью определяется начальными условиями и не изменяется со временем.

---

## Входные параметры

```python
def kepler_propagate(r0, v0, dt_days, mu):
```

| Параметр | Описание | Единицы |
|----------|---------|---------|
| **r0** | Начальное гелиоцентрическое положение объекта | AU (астрономические единицы) |
| **v0** | Начальная гелиоцентрическая скорость объекта | AU/день |
| **dt_days** | Интервал времени интегрирования | дни |
| **mu** | Гравитационный параметр Солнца (μ = k²) | AU³/день² |

---

## Шаг 1. Приведение к numpy-массивам

```python
r0 = np.array(r0, dtype=float)
v0 = np.array(v0, dtype=float)
```

**Смысл:** Преобразуем входные данные в numpy-массивы типа `float64` для последующей численной работы.

---

## Шаг 2. Вычисление модулей (норм) векторов

```python
r0_norm = vector_norm(r0)
v0_norm = vector_norm(v0)
```

**Формулы:**
$$r_0 = |\mathbf{r}_0| = \sqrt{r_{0x}^2 + r_{0y}^2 + r_{0z}^2}$$

$$v_0 = |\mathbf{v}_0| = \sqrt{v_{0x}^2 + v_{0y}^2 + v_{0z}^2}$$

**Геометрический смысл:**
- **r₀** — расстояние от Солнца до объекта в начальный момент
- **v₀** — скорость объекта (или её модуль)

**Где используются:** В формулах для энергии, большой полуоси, эксцентриситета.

---

## Шаг 3. Проверка корректности начальных данных

```python
if r0_norm < 1e-12:
    return r0, v0
```

**Смысл:** Если начальное расстояние меньше ~10⁻¹² AU, это некорректные данные (объект практически в центре Солнца). Возвращаем исходные данные как есть.

---

## Шаг 4. Вычисление удельной орбитальной энергии

```python
energy = v0_norm ** 2 / 2.0 - mu / r0_norm
```

**Формула:**
$$\varepsilon = \frac{v_0^2}{2} - \frac{\mu}{r_0}$$

где:
- $\frac{v_0^2}{2}$ — удельная кинетическая энергия
- $-\frac{\mu}{r_0}$ — удельная потенциальная энергия гравитационного поля

**Физический смысл:**
- Удельная энергия **постоянна** вдоль орбиты (закон сохранения энергии)
- Она определяет, является ли орбита эллипсом, параболой или гиперболой:
  - $\varepsilon < 0$ → эллипсическая орбита (замкнутая)
  - $\varepsilon = 0$ → параболическая орбита (на пределе)
  - $\varepsilon > 0$ → гиперболическая орбита (объект улетит в бесконечность)

**Проверка:**
```python
if energy >= 0:
    return r0, v0
```

Если энергия неотрицательна, орбита не эллиптическая, и данный интегратор (рассчитанный на эллипсы) не работает → возвращаем исходные значения.

---

## Шаг 5. Вычисление большой полуоси

```python
a = -mu / (2.0 * energy)
```

**Формула:**
$$a = -\frac{\mu}{2\varepsilon}$$

**Вывод:** Из условия сохранения энергии. Для эллипса $\varepsilon < 0$, поэтому $a > 0$.

**Геометрический смысл:**
- **a** — половина длины большой оси эллипса
- Это характеристический размер орбиты
- Определяет период обращения (третий закон Кеплера): $T = 2\pi\sqrt{\frac{a^3}{\mu}}$

---

## Шаг 6. Вычисление вектора углового момента

```python
h_vec = np.cross(r0, v0)
h = vector_norm(h_vec)
```

**Формула:**
$$\mathbf{h} = \mathbf{r}_0 \times \mathbf{v}_0$$

$$h = |\mathbf{h}| = r_0 \cdot v_0 \cdot \sin\theta$$

где θ — угол между **r₀** и **v₀**.

**Физический смысл:**
- **h** — удельный угловой момент (момент импульса на единицу массы)
- Он постоянен вдоль орбиты (закон сохранения углового момента)
- **h** определяет ориентацию плоскости орбиты (нормальный вектор к плоскости)
- $h = a \cdot b \cdot n$, где b — малая полуось, n — среднее движение

**Проверка:**
```python
if h < 1e-12:
    return r0, v0
```

Если h ≈ 0, движение радиальное (нет углового момента). Орбита вырождается в линию через Солнце. Метод не применим.

---

## Шаг 7. Вычисление вектора эксцентриситета

```python
e_vec = (np.cross(v0, h_vec) / mu) - (r0 / r0_norm)
e = vector_norm(e_vec)
```

**Формулы:**

$$\mathbf{e}_{vec} = \frac{\mathbf{v}_0 \times \mathbf{h}}{\mu} - \frac{\mathbf{r}_0}{r_0}$$

$$e = |\mathbf{e}_{vec}|$$

**Физический смысл:**
- **e** — эксцентриситет орбиты
  - e = 0 → круговая орбита
  - 0 < e < 1 → эллипсическая орбита
  - e = 1 → параболическая орбита
  - e > 1 → гиперболическая орбита
  
- **e**_vec — вектор эксцентриситета, направлен в сторону перицентра (ближайшей к Солнцу точки орбиты)
- Длина этого вектора = e

**Геометрический смысл:** Вектор **e** указывает направление наибольшего приближения объекта к Солнцу.

---

## Шаг 8. Ограничение эксцентриситета

```python
if e >= 0.99:
    e = 0.99
    e_vec = e_vec * (e / vector_norm(e_vec))
```

**Смысл:** Для предотвращения численных нестабильностей ограничиваем e максимумом 0.99. Это защищает от орбит, близких к параболическим (e ≈ 1).

---

## Шаг 9. Особый случай — почти круговая орбита (e < 10⁻⁸)

```python
if e < 1e-8:
```

Для круговых орбит используется упрощённый алгоритм.

### Шаг 9a. Среднее движение

```python
n = np.sqrt(mu / a ** 3)
M = n * dt_days
```

**Формулы:**

$$n = \sqrt{\frac{\mu}{a^3}}$$  (среднее движение, угловая частота)

$$M = n \cdot \Delta t$$  (средняя аномалия, угол, пройденный за время Δt)

**Физический смысл:**
- **n** — угловая скорость орбиты для круговой траектории
- **M** — угол, на который объект повернулся около Солнца за время Δt

### Шаг 9b. Базис в плоскости орбиты

```python
k_vec = np.array([0.0, 0.0, 1.0])
i_vec = np.cross(k_vec, h_vec)
```

**Геометрический смысл:**
- **k** — единичный вектор вдоль оси Z (нормаль к эклиптике)
- **i** — направление пересечения плоскости орбиты с плоскостью эклиптики (восходящий узел)
- **i** = k × h — перпендикулярно и k, и h

```python
i_norm = np.linalg.norm(i_vec)
if i_norm < 1e-15:
    i_vec = np.array([1.0, 0.0, 0.0])
else:
    i_vec /= i_norm
```

Нормируем **i** (приводим к единичной длине). Если норма < 10⁻¹⁵, орбита практически экваториальна (h почти вдоль Z), выбираем произвольное направление **i** = (1, 0, 0).

```python
j_vec = np.cross(h_vec/h, i_vec)
```

**j** = (h/|h|) × **i** — третий орт базиса (дополняет **i** в плоскости орбиты).

Теперь (**i**, **j**, h/|h|) — ортонормированный базис плоскости орбиты.

### Шаг 9c. Проекция начального положения

```python
x0 = np.dot(r0, i_vec)
y0 = np.dot(r0, j_vec)
r = np.sqrt(x0**2 + y0**2)
phi0 = np.arctan2(y0, x0)
```

**Формулы:**

$$x_0 = \mathbf{r}_0 \cdot \mathbf{i}$$  (проекция на ось перицентра)

$$y_0 = \mathbf{r}_0 \cdot \mathbf{j}$$  (проекция на перпендикулярное направление)

$$r = \sqrt{x_0^2 + y_0^2}$$  (радиус в плоскости орбиты, для круга = a)

$$\varphi_0 = \text{atan2}(y_0, x_0)$$  (начальный угол в плоскости орбиты)

### Шаг 9d. Новый угол и положение

```python
phi = phi0 + M
r_vec = r * (np.cos(phi)*i_vec + np.sin(phi)*j_vec)
```

**Формула:**

$$\varphi = \varphi_0 + M$$

$$\mathbf{r}_{new} = r \cdot (\cos\varphi \cdot \mathbf{i} + \sin\varphi \cdot \mathbf{j})$$

**Геометрический смысл:** В круговой орбите объект просто вращается с постоянной угловой скоростью n. За время Δt он повернулся на угол M. Новое положение получается поворотом начального положения на этот угол в плоскости орбиты.

```python
return r_vec, v0
```

Скорость остаётся той же (упрощённое приближение для почти круговых орбит).

---

## Шаг 10. Общий случай — эллиптическая орбита (0 < e < 1)

### Шаг 10a. Начальная истинная аномалия

```python
cos_nu0 = np.dot(e_vec, r0) / (e * r0_norm)
cos_nu0 = np.clip(cos_nu0, -1.0, 1.0)
nu0 = np.arccos(cos_nu0)
if np.dot(r0, v0) < 0:
    nu0 = 2 * np.pi - nu0
```

**Формула:**

$$\cos \nu_0 = \frac{\mathbf{e} \cdot \mathbf{r}_0}{e \cdot r_0}$$

$$\nu_0 = \arccos\left(\frac{\mathbf{e} \cdot \mathbf{r}_0}{e \cdot r_0}\right)$$

**Физический смысл:**
- **ν₀** — истинная аномалия в начальный момент
- Это угол между вектором эксцентриситета **e** и радиус-вектором **r₀**, измеренный из центра Солнца
- Если **r₀** · **v₀** < 0, объект движется "назад" (от апоцентра к перицентру), поэтому ν₀ ∈ (π, 2π)

**Клипирование:** `np.clip(cos_nu0, -1, 1)` защищает от численных ошибок, которые могут дать cos(ν) > 1 или < -1.

### Шаг 10b. Переход к эксцентриситетной аномалии

```python
arg = (1 - e) / (1 + e)
if arg <= 0:
    return r0, v0

E0 = 2 * np.arctan(np.tan(nu0 / 2) * np.sqrt(arg))
```

**Формула:**

$$E_0 = 2 \arctan\left(\tan\frac{\nu_0}{2} \sqrt{\frac{1-e}{1+e}}\right)$$

Это **формула Лэндау** для связи между истинной (ν) и эксцентриситетной (E) аномалиями.

**Геометрический смысл:**
- **E** — эксцентриситетная аномалия
- Это вспомогательный угол, введённый для удобства решения уравнения Кеплера
- Связан с истинной аномалией формулой выше
- Диапазон: если ν ∈ [0, 2π], то E ∈ [0, 2π]

### Шаг 10c. Начальная средняя аномалия

```python
M0 = E0 - e * np.sin(E0)
```

**Формула (уравнение Кеплера):**

$$M_0 = E_0 - e \sin E_0$$

где:
- **M** — средняя аномалия
- Это угол, который объект прошёл бы, если бы двигался с постоянной угловой скоростью

**Смысл:** Уравнение Кеплера связывает трансцендентным образом три аномалии: M, E, ν. Оно не имеет замкнутого решения, поэтому обычно решается численно.

### Шаг 10d. Проверка большой полуоси

```python
if a <= 0:
    return r0, v0
```

Защита от некорректных значений большой полуоси.

### Шаг 10e. Среднее движение и конечная средняя аномалия

```python
n = np.sqrt(mu / a ** 3)
M = M0 + n * dt_days
```

**Формула:**

$$M = M_0 + n \cdot \Delta t$$

где $n = \sqrt{\frac{\mu}{a^3}}$ — среднее движение.

**Физический смысл:** Средняя аномалия растёт линейно со временем. За время Δt она увеличивается на n·Δt.

---

### Шаг 10f. Решение уравнения Кеплера

```python
E = M.copy()
for _ in range(30):
    f = E - e * np.sin(E) - M
    fp = 1 - e * np.cos(E)
    dE = -f / fp
    E += dE
    if np.max(np.abs(dE)) < 1e-12:
        break
```

**Метод:** Метод Ньютона для уравнения $f(E) = E - e\sin E - M = 0$

**Формулы:**

$$f(E) = E - e \sin E - M$$

$$f'(E) = 1 - e \cos E$$

$$E_{new} = E - \frac{f(E)}{f'(E)} = E - \frac{E - e\sin E - M}{1 - e\cos E}$$

**Алгоритм:**
1. Начальное приближение: E = M
2. Итерируем до сходимости (пока |ΔE| < 10⁻¹²)
3. Обычно сходится за 3-5 итераций

**Смысл:** Решаем уравнение Кеплера, чтобы найти эксцентриситетную аномалию E в момент времени t = t₀ + Δt.

---

### Шаг 10g. Вычисление расстояния

```python
cos_E = np.cos(E)
r = a * (1 - e * cos_E)
```

**Формула:**

$$r = a(1 - e \cos E)$$

**Вывод:** Из определения эксцентриситетной аномалии в декартовых координатах.

**Геометрический смысл:**
- На перицентре (E = 0): r = a(1 - e) — минимальное расстояние
- На апоцентре (E = π): r = a(1 + e) — максимальное расстояние
- Для круга (e = 0): r = a (постоянно)

---

### Шаг 10h. Базис плоскости орбиты (повторно)

```python
k_vec = np.array([0.0, 0.0, 1.0])
i_vec = np.cross(k_vec, h_vec)
i_norm = vector_norm(i_vec)
if i_norm < 1e-15:
    i_vec = np.array([1.0, 0.0, 0.0])
else:
    i_vec /= i_norm
j_vec = np.cross(h_vec/h, i_vec)
```

Аналогично шагу 9b, создаём ортонормированный базис плоскости орбиты.

### Шаг 10i. Вектор эксцентриситета как направление перицентра

```python
if e > 1e-10:
    p_dir = e_vec / e
else:
    p_dir = i_vec
q_dir = np.cross(h_vec / h, p_dir)
```

**Геометрический смысл:**
- **p_dir** — единичный вектор в направлении перицентра (вдоль вектора **e**)
- **q_dir** — единичный вектор, перпендикулярный **p_dir** в плоскости орбиты

Вместе они образуют естественный базис (p, q) для орбиты.

### Шаг 10j. Истинная аномалия через эксцентриситетную

```python
factor = (1 + e) / (1 - e)
if factor <= 0:
    return r0, v0
nu = 2 * np.arctan2(np.sqrt(factor) * np.sin(E/2), np.cos(E/2))
```

**Формула:**

$$\nu = 2 \arctan\left(\sqrt{\frac{1+e}{1-e}} \tan\frac{E}{2}\right)$$

или эквивалентно:

$$\nu = 2 \arctan2\left(\sqrt{\frac{1+e}{1-e}} \sin\frac{E}{2}, \cos\frac{E}{2}\right)$$

**Смысл:** Преобразуем эксцентриситетную аномалию E (которую мы нашли из уравнения Кеплера) обратно в истинную аномалию ν.

**Функция `arctan2`:** Возвращает угол в диапазоне (-π, π], правильно учитывая квадрант.

---

### Шаг 10k. Новое положение в плоскости орбиты

```python
r_vec = r * (np.cos(nu) * p_dir + np.sin(nu) * q_dir)
```

**Формула (полярные координаты в плоскости орбиты):**

$$\mathbf{r}_{new} = r \cdot (\cos\nu \cdot \mathbf{p}_{dir} + \sin\nu \cdot \mathbf{q}_{dir})$$

**Геометрический смысл:**
- В плоскости орбиты используем полярные координаты (r, ν)
- **p_dir** — радиальное направление (к перицентру)
- **q_dir** — тангенциальное направление (перпендикулярно радиусу)
- Истинная аномалия ν — угол от перицентра

---

### Шаг 10l. Компоненты скорости

```python
rdot = (mu / h) * e * np.sin(nu)
rfdot = (mu / h) * (1 + e * np.cos(nu))
v_vec = rdot * (r_vec / r) + rfdot * q_dir
```

**Формулы:**

$$\dot{r} = \frac{\mu}{h} e \sin \nu$$  (радиальная компонента скорости)

$$r\dot{f} = \frac{\mu}{h} (1 + e \cos \nu)$$  (тангенциальная компонента, умноженная на r)

$$\mathbf{v} = \dot{r} \frac{\mathbf{r}}{r} + \dot{f} \mathbf{q}_{dir}$$

**Вывод:** Из уравнений движения в полярных координатах в гравитационном поле с обратным квадратом закона.

**Физический смысл:**
- На перицентре (ν = 0): $\dot{r} = 0$, максимальная скорость (объект движется только перпендикулярно радиусу)
- На апоцентре (ν = π): $\dot{r} = 0$, минимальная скорость
- На средних точках орбиты: обе компоненты ненулевые

---

## Выходные параметры

```python
return r_vec, v_vec
```

| Возвращаемое | Описание | Единицы |
|--------------|---------|---------|
| **r_vec** | Гелиоцентрическое положение в момент t₀ + Δt | AU |
| **v_vec** | Гелиоцентрическая скорость в момент t₀ + Δt | AU/день |

---

## Схема алгоритма

```
┌─────────────────────────────────────────────┐
│ Входные данные: r₀, v₀, Δt, μ              │
└──────────────────┬──────────────────────────┘
                   │
                   ▼
         ┌─────────────────────┐
         │ Вычисляем r₀, v₀    │
         │ Вычисляем ε (энергия)│
         └────────┬────────────┘
                  │
          ┌───────▼───────┐
          │ ε < 0?        │
          └───┬───────┬───┘
         НЕТ │       │ ДА
             │   ▼   │
             │ (exit) │
             │       │ ▼
             │   ┌────────────────────┐
             │   │ Вычисляем a, h, e  │
             │   │ Вычисляем e_vec    │
             │   └────┬───────────────┘
             │        │
             │   ┌────▼────┐
             │   │ e < 10⁻⁸?│
             │   └─┬───┬───┘
             │  ДА │   │ НЕТ
             │    ▼   └─────────────────┐
             │ (КРУГ)         (ЭЛЛИПС) │
             │  │                      │
             │  │ Упрощённый      ┌────▼────────┐
             │  │ алгоритм        │ Решаем      │
             │  │                 │ уравнение   │
             │  │                 │ Кеплера     │
             │  │                 │ методом     │
             │  │                 │ Ньютона     │
             │  │                 └────┬────────┘
             │  │                      │
             │  │  ┌──────────────────┘
             │  │  │
             │  └──┴─────────────────┐
             │                       │
             │ ┌─────────────────────▼──┐
             │ │ Вычисляем ν и r       │
             │ │ Вычисляем компоненты v│
             │ │ Преобразуем в 3D      │
             │ └─────────────────┬──────┘
             │                   │
             └───────────┬───────┘
                         │
                         ▼
            ┌──────────────────────┐
            │ Выход: r, v          │
            └──────────────────────┘
```

---

## Пример с числами

Предположим:
- r₀ = (0.7, 0.3, 0.1) AU
- v₀ = (-0.01, 0.015, 0.002) AU/день
- Δt = 100 дней
- μ = 1.0 (для Солнца в астрономических единицах)

**Шаг 1:** r₀_norm = √(0.7² + 0.3² + 0.1²) ≈ 0.775 AU

**Шаг 2:** v₀_norm = √(0.01² + 0.015² + 0.002²) ≈ 0.018 AU/день

**Шаг 3:** ε = 0.018²/2 - 1.0/0.775 ≈ 0.000162 - 1.290 ≈ -1.290 (эллипс ✓)

**Шаг 4:** a = -1.0 / (2 × (-1.290)) ≈ 0.388 AU

**Шаг 5:** **h** = r₀ × v₀ ≈ (0.0008, -0.0003, 0.0115) AU²/день
           h ≈ 0.0115 AU²/день

**Шаг 6:** e ≈ 0.72 (вытянутый эллипс)

**Шаг 7-10:** Решаем уравнение Кеплера, получаем E ≈ 4.15 рад, ν ≈ 4.22 рад

**Шаг 11:** r ≈ 0.388(1 - 0.72 cos(4.15)) ≈ 0.550 AU

**Вывод:** r_new ≈ (0.15, 0.52, 0.08) AU, v_new ≈ (-0.018, -0.005, 0.001) AU/день

---

## Ограничения и точность

| Ограничение | Причина |
|------------|---------|
| e < 0.99 | Численная стабильность близ параболы |
| max 30 итераций Ньютона | Обычно сходится за 3-5, 30 — достаточный запас |
| Точность 10⁻¹² | Двойная точность float64 |
| Эллиптические орбиты только | Параболы и гиперболы требуют других методов |

---

## Применение в методе Гаусса–Ньютона

В контексте уточнения орбиты:

1. **Инициализация:** x₀ = (r₂, v₂) из метода Гаусса
2. **Итерация:** Для каждого наблюдения вычисляем прогноз:
   ```
   r_prog, v_prog = kepler_propagate(r2, v2, dt, μ)
   direction_model = (r_prog - R_obs) / |r_prog - R_obs|
   ```
3. **Невязка:** residual = direction_model - direction_observed
4. **Минимизация:** Гаусс–Ньютон минимизирует сумму квадратов невязок

Функция `kepler_propagate` вызывается сотни раз, поэтому её эффективность критична.

